# Слияние таблиц, дубликаты и противоречия

Объединение (union) двух таблиц с разной структурой, внутреннее и левое соединение, затем поиск полных дубликатов и противоречий: записей, у которых ключ совпадает, а значения расходятся

In [1]:
import pandas as pd

#Таблица 3.13
table_13 = pd.DataFrame({
    "Поставщик": [
        "ООО «Невод»", "ЗАО «Молкомбинат»", "ООО «Птичник»",
        "ЗАО «Овощевод»", "ОАО «Хладокомбинат»", "ЗАО «Маслосырбаза»",
        "ОАО «Горпищекомбинат»"
    ],
    "Товар": [
        "Рыба свежая", "Сыр", "Куры", "Капуста",
        "Мороженое", "Молоко", "Кетчуп"
    ]
})

#Таблица 3.14
table_14 = pd.DataFrame({
    "Дата": [
        "02.04.2024","08.04.2024","11.04.2024",
        "15.04.2024","22.04.2024","27.04.2024","27.04.2024"
    ],
    "Поставщик": [
        "ЗАО «Молкомбинат»","ООО «Птичник»","ЗАО «Маслосырбаза»",
        "ООО «Птичник»","ЗАО «Молкомбинат»","ОАО «Хладокомбинат»",
        "ООО «Невод»"
    ],
    "Товар": [
        "Сыр","Куры","Молоко","Куры","Сыр","Мороженое","Рыба свежая"
    ],
    "Количество": [150,200,170,100,80,250,160]
})

In [2]:
#Добавляем в первую таблицу недостающие поля
table_13_ext = table_13.copy()
table_13_ext["Дата"] = None
table_13_ext["Количество"] = None

#Выравниваем порядок столбцов
table_13_ext = table_13_ext[["Поставщик","Товар","Дата","Количество"]]
table_14_ext = table_14[["Поставщик","Товар","Дата","Количество"]]

#Объединение (union)
union_result = pd.concat([table_13_ext, table_14_ext], ignore_index=True)

print(union_result)

                Поставщик        Товар        Дата Количество
0             ООО «Невод»  Рыба свежая        None       None
1       ЗАО «Молкомбинат»          Сыр        None       None
2           ООО «Птичник»         Куры        None       None
3          ЗАО «Овощевод»      Капуста        None       None
4     ОАО «Хладокомбинат»    Мороженое        None       None
5      ЗАО «Маслосырбаза»       Молоко        None       None
6   ОАО «Горпищекомбинат»       Кетчуп        None       None
7       ЗАО «Молкомбинат»          Сыр  02.04.2024        150
8           ООО «Птичник»         Куры  08.04.2024        200
9      ЗАО «Маслосырбаза»       Молоко  11.04.2024        170
10          ООО «Птичник»         Куры  15.04.2024        100
11      ЗАО «Молкомбинат»          Сыр  22.04.2024         80
12    ОАО «Хладокомбинат»    Мороженое  27.04.2024        250
13            ООО «Невод»  Рыба свежая  27.04.2024        160


In [3]:
inner_join = pd.merge(table_13, table_14, on=["Поставщик", "Товар"], how="inner")

print(inner_join)

print("-------------------------------------------------------------------------")

left_join = pd.merge(table_13, table_14, on=["Поставщик", "Товар"], how="left")

print(left_join)

             Поставщик        Товар        Дата  Количество
0   ЗАО «Маслосырбаза»       Молоко  11.04.2024         170
1    ЗАО «Молкомбинат»          Сыр  02.04.2024         150
2    ЗАО «Молкомбинат»          Сыр  22.04.2024          80
3  ОАО «Хладокомбинат»    Мороженое  27.04.2024         250
4        ООО «Птичник»         Куры  15.04.2024         100
5        ООО «Птичник»         Куры  08.04.2024         200
6          ООО «Невод»  Рыба свежая  27.04.2024         160
-------------------------------------------------------------------------
               Поставщик        Товар        Дата  Количество
0            ООО «Невод»  Рыба свежая  27.04.2024       160.0
1      ЗАО «Молкомбинат»          Сыр  02.04.2024       150.0
2      ЗАО «Молкомбинат»          Сыр  22.04.2024        80.0
3          ООО «Птичник»         Куры  08.04.2024       200.0
4          ООО «Птичник»         Куры  15.04.2024       100.0
5         ЗАО «Овощевод»      Капуста         NaN         NaN
6    ОАО «Хл

In [4]:
table_18 = pd.DataFrame({
    "Поле 1": ["01.01.2004","21.05.2004","21.05.2004","21.05.2004","01.09.2004","01.09.2004"],
    "Поле 2": [2,3,3,3,4,4],
    "Поле 3": [1000,1000,700,700,1200,1200],
    "Поле 4": [1500,1500,1500,1500,1700,1700]
})

duplicates = table_18[table_18.duplicated()]
print("Дубликаты:")
print(duplicates)


Дубликаты:
       Поле 1  Поле 2  Поле 3  Поле 4
3  21.05.2004       3     700    1500
5  01.09.2004       4    1200    1700


In [5]:
#Группировка по ключу и поиск различий
conflicts = table_18.groupby(["Поле 1", "Поле 2"]).filter(lambda g: len(g["Поле 3"].unique()) > 1 or len(g["Поле 4"].unique()) > 1)

print("Противоречия:")
print(conflicts)

Противоречия:
       Поле 1  Поле 2  Поле 3  Поле 4
1  21.05.2004       3    1000    1500
2  21.05.2004       3     700    1500
3  21.05.2004       3     700    1500
